# RealWaste Progres 4 — Phase 3 Controlled Ablation Study (224x224 & 2-Stage Fine-Tuning)
Menarik kode modular `src/` beserta hasil `EXP-01..EXP-07` dari GitHub, membangun cache `uint8` **224x224**, menyimpan **Sanity Check Batch Grid**, menjalankan **EXP-08..EXP-11** pada GPU Tesla T4 Kaggle, serta menghasilkan evaluasi & audit kalibrasi lengkap pada **Validation Set**.

In [ ]:
import os, sys, platform, torch
print('Python:', platform.python_version())
print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))


In [ ]:
REPO_URL = 'https://github.com/tryaannn/klasifikasi-jenis-sampah.git'
BRANCH = 'test/majid'
REPO_DIR = '/kaggle/working/klasifikasi-jenis-sampah'

!rm -rf {REPO_DIR}
!git clone --branch {BRANCH} --depth 1 {REPO_URL} {REPO_DIR}
%cd {REPO_DIR}
!git log -n 2 --oneline


In [ ]:
# Verifikasi unit test & smoke test sebelum training GPU
!python -m pytest -q

from pathlib import Path
from src.dataset import resolve_realwaste_root
rw_root = resolve_realwaste_root('/kaggle/input')
print('Resolved RealWaste root:', rw_root)


In [ ]:
import shutil
CACHE_224 = '/kaggle/working/realwaste_224_uint8.npz'
OUT_RESULTS = '/kaggle/working/progress4_results'
OUT_FIGURES = '/kaggle/working/progress4_figures'

# Salin hasil EXP-01..EXP-07 agar comparison table mencakup semua eksperimen
shutil.copytree('results/progress4', OUT_RESULTS, dirs_exist_ok=True)
!python -m src.dataset --data-dir "{rw_root}" --image-size 224 --cache-out "{CACHE_224}"


In [ ]:
# Eksekusi Ablation Study EXP-08..EXP-11 (224x224, 2-Stage FT, Stage 2 Early Stopping patience=7)
!python -m src.train --experiments ablation224 \
    --data-dir "{rw_root}" \
    --cache-path "{CACHE_224}" \
    --output-dir "{OUT_RESULTS}" \
    --figures-dir "{OUT_FIGURES}" \
    --patience 7 --seed 42


In [ ]:
# Evaluasi checkpoint terbaik KHUSUS pada Validation Set (713 citra) + Audit Kalibrasi & Error Analysis
!python -m src.evaluate_val \
    --results-dir "{OUT_RESULTS}" \
    --figures-dir "{OUT_FIGURES}" \
    --data-dir "{rw_root}" \
    --cache-path "{CACHE_224}"


In [ ]:
import json, os
from pathlib import Path
%cd /kaggle/working
if os.path.exists(CACHE_224):
    os.remove(CACHE_224)
!rm -rf /kaggle/working/klasifikasi-jenis-sampah

comp = json.loads(Path(f'{OUT_RESULTS}/experiment_comparison.json').read_text())
print(json.dumps(comp, indent=2))
